# Experiment 01 - Lab Setup
**Author:** Austin Howard \
**Date:** November 13, 2026 \
**Project:** Master's thesis research in adversarial audio \
**Repository:** ADH-Cyber/adversarial-audio

## 1. Objective
A baseline computational will serve as a foundation for future experimentation for both GAN- and Diffusion-based adversarial perturbation generation. In order to decrease Wall-clock time calculation after each time an experiment is run, GPU acceleration will be utilized in order to maximize parallel computation capabilities. This will be exceptionally helpful for tensor operations like matrix multiplication.

A successful lab setup will allow us to utilize our GPU for processing these calculations. In particular, we need:
- PyTorch to recognize the NVIDIA GPU.
- CUDA to initialize successfully.
- Tensor operations to execute on GPU without errors.

## Environment installation

Select the intended Python environment as the notebook kernel. For GPU acceleration, first install a PyTorch build appropriate for your operating system and CUDA setup using the [official PyTorch installation selector](https://pytorch.org/get-started/locally/). Then run the cell below from the `notebooks` directory. It installs the direct dependencies used by Experiments 1–4 into the active kernel. The requirements file is not a lock file; record the resolved versions for each research run. Restart the kernel after installation, then run the remaining cells. Model weights are downloaded separately on first use.

In [ ]:
%pip install -r ../requirements.txt

## 2. Environment
This section documents the hardware and software configuraiton being used to establish the experimental environment. This will allow for reproducible results and identify computation constraints.

### 2.1 Hardware configuration
The configuration of hardware determines the computational resources available for model development. GPU architecture and memory capacity are of partiular relevance, they influence model compatibility and performance. The following code will retrieve pertinent system information:

In [9]:
import platform
import subprocess
import torch
from IPython.display import display, Markdown

def powershell(command):
    try:
        return subprocess.check_output(
            ["powershell", "-NoProfile", "-Command", command],
            text=True
        ).strip()
    except (OSError, subprocess.CalledProcessError):
        return "Unavailable"

cpu = powershell(
    "(Get-CimInstance Win32_Processor).Name"
)

ram = powershell(
    "[math]::Round((Get-CimInstance Win32_ComputerSystem).TotalPhysicalMemory / 1GB, 2)"
)

hardware = {
    "CPU": cpu,
    "System RAM": f"{ram} GB",
    "GPU": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Unavailable",
    "GPU VRAM": (
        f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GiB"
        if torch.cuda.is_available() else "Unavailable"
    ),
    "CUDA Compute Capability": (
        ".".join(map(str, torch.cuda.get_device_capability(0)))
        if torch.cuda.is_available() else "Unavailable"
    )
}

table = "| Component | Specification |\n|---|---|\n"
for component, value in hardware.items():
    table += f"| {component} | {value} |\n"

display(Markdown(table))


| Component | Specification |
|---|---|
| CPU | Intel(R) Core(TM) Ultra 9 285H |
| System RAM | 31.43 GB |
| GPU | NVIDIA GeForce RTX 5080 Laptop GPU |
| GPU VRAM | 15.89 GiB |
| CUDA Compute Capability | 12.0 |


### 2.2 Software Configuration
The software environment is centered around Python, PyTorch, CUDA, and their supporting devkits which enable GPU acceleration for machine learning experimentation. The code executed below will retrive the installed software versions and the active Python environment, showing the configuration at the time of execution.

In [11]:

import sys
import platform
import subprocess
import importlib.metadata as metadata
import torch
from IPython.display import display, Markdown

def version(package):
    try:
        return metadata.version(package)
    except metadata.PackageNotFoundError:
        return "Not installed"

def command_output(command):
    try:
        return subprocess.check_output(
            command,
            text=True,
            stderr=subprocess.DEVNULL
        ).strip()
    except (OSError, subprocess.CalledProcessError):
        return "Unavailable"

software = {
    "Operating System": platform.platform(),
    "Python": platform.python_version(),
    "PyTorch": torch.__version__,
    "TorchVision": version("torchvision"),
    "TorchAudio": version("torchaudio"),
    "NumPy": version("numpy"),
    "CUDA Runtime (PyTorch)": torch.version.cuda,
    "NVIDIA Driver": command_output([
        "nvidia-smi",
        "--query-gpu=driver_version",
        "--format=csv,noheader"
    ]).splitlines()[0],
    "JupyterLab": version("jupyterlab"),
    "IPython Kernel": version("ipykernel"),
    "Visual Studio Code": command_output(["code", "--version"]).splitlines()[0],
    "Git": command_output(["git", "--version"]),
    "Python Environment": sys.prefix
}

table = "| Component | Version / Configuration |\n|---|---|\n"
for component, value in software.items():
    table += f"| {component} | {value} |\n"

display(Markdown(table))


| Component | Version / Configuration |
|---|---|
| Operating System | Windows-11-10.0.26300-SP0 |
| Python | 3.14.9 |
| PyTorch | 2.15.0+cu132 |
| TorchVision | 0.29.2+cu132 |
| TorchAudio | 2.11.1+cu132 |
| NumPy | 2.5.4 |
| CUDA Runtime (PyTorch) | 13.2 |
| NVIDIA Driver | 621.24 |
| JupyterLab | 4.6.6 |
| IPython Kernel | 7.4.0 |
| Visual Studio Code | Unavailable |
| Git | git version 2.56.0.windows.3 |
| Python Environment | C:\Users\Austin\Documents\Code\Adversarial Audio\adversarial-audio\gan.venv |


## 3. Validation
Recall our success criteria:
- PyTorch to recognize the NVIDIA GPU.
- CUDA to initialize successfully.
- Tensor operations to execute on GPU without errors.

We'll test those in the sections below.

### 3.1 GPU Detection
We first verify that PyTorch recognizes the NVIDIA GPU:

In [12]:
import torch

print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Compute Capability:", torch.cuda.get_device_capability(0))


CUDA Available: True
GPU: NVIDIA GeForce RTX 5080 Laptop GPU
Compute Capability: (12, 0)


**Success:** CUDA is available and the correct GPU is identified.

### 3.2 CUDA Initialization
Now, we'll verify that PyTorch can initialize CUDA and allocate a tensor in GPU memory:

In [13]:
torch.cuda.init()

x = torch.tensor([1.0, 2.0, 3.0], device="cuda")

print("CUDA Initialized:", torch.cuda.is_initialized())
print("Tensor Device:", x.device)


CUDA Initialized: True
Tensor Device: cuda:0


**Success:** CUDA has initilized and the tensor is allocated on `cuda:0`.

### 3.3 GPU Tensor Computation
We must verify that PyTorch can execute matrix multiplication on the GPU.

In [14]:
a = torch.randn(1000, 1000, device="cuda")
b = torch.randn(1000, 1000, device="cuda")

result = torch.matmul(a, b)
torch.cuda.synchronize()

print("Result Shape:", result.shape)
print("Result Device:", result.device)
print("Computation Successful:", torch.isfinite(result).all().item())


Result Shape: torch.Size([1000, 1000])
Result Device: cuda:0
Computation Successful: True


**Success:** The operation has completed without errors. It produced a $1000 \times 1000$ tensor on the GPU and only contains finite values.

## 4. Conclusion
We have successfully installed our lab environment and can begin creating consistently reproducible experiments.
**Next lab:** Generate adversarial perturbations using an existing API.

In [12]:
!jupyter nbconvert --to html --template lab "Exp1-LabSetup.ipynb"

[NbConvertApp] Converting notebook Exp1-LabSetup.ipynb to html
[NbConvertApp] Writing 332286 bytes to Exp1-LabSetup.html
